# Importing Important Libraries For Extraction and Transformation of Data

In [46]:
import pandas as pd
import numpy as np
import fastf1
import os

## Caching Data from Fast F1

In [47]:
fastf1.Cache.enable_cache("f1_cache")

### Loading Data of 2023,2024,2025 Austrain Grand Prix

In [48]:
os.makedirs("f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")

sessions = {}

for year in range(2023, 2026):
    try:
        session = fastf1.get_session(year, 'Austria', 'R')
        session.load()
        sessions[year] = session
        print(f"{year} loaded successfully")
    except Exception as e:
        print(f"Failed for {year}: {e}")

core           INFO 	Loading data for Austrian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '11', '4', '14', '55', '63', '44', '18', '10', '23', '24', '2', '31', '77', '81', '21', '20', '22', '27']


2023 loaded successfully


core           INFO 	Loading data for Austrian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['63', '81', '55', '44', '1', '27', '11', '20', '3', '10', '16', '31', '18', '22', '23', '77', '24', '14', '2', '4']


2024 loaded successfully


core           INFO 	Loading data for Austrian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '16', '44', '63', '30', '14', '5', '27', '31', '87', '6', '10', '18', '43', '22', '23', '1', '12', '55']


2025 loaded successfully


## Fecthing important Columns

In [49]:
laps={}
for year in range(2023, 2026):
    
    laps[year] = sessions[year].laps[["Driver","LapTime","Sector1Time","Sector2Time","Sector3Time"]].copy()
    

In [50]:
laps[2023].head()

,Driver,LapTime,Sector1Time,Sector2Time,Sector3Time
0,VER,0 days 00:01:17.639000,NaT,0 days 00:00:31.613000,0 days 00:00:25.440000
1,VER,0 days 00:01:55.479000,0 days 00:00:31.698000,0 days 00:00:46.293000,0 days 00:00:37.488000
2,VER,0 days 00:02:04.721000,0 days 00:00:37.877000,0 days 00:00:46.298000,0 days 00:00:40.546000
3,VER,0 days 00:01:09.691000,0 days 00:00:17.618000,0 days 00:00:30.970000,0 days 00:00:21.103000
4,VER,0 days 00:01:10.026000,0 days 00:00:17.716000,0 days 00:00:31.158000,0 days 00:00:21.152000


## Dropping all the Null Values


In [51]:
for year in range (2023,2026):
    laps[year].dropna(inplace=True)

## Converting the Times in Seconds 

In [52]:
for year in range (2023,2026):
    for col in["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
        laps[year][f"{col} (s)"] = laps[year][col].dt.total_seconds()

## Taking Average of all the Times

In [53]:
sector_times = {}

for year in range(2023,2026):
    sector_times[year] = laps[year].groupby("Driver").agg({
        "Sector1Time (s)":"mean",
        "Sector2Time (s)":"mean",
        "Sector3Time (s)":"mean"
    }).reset_index()

## Add New Columns Like Total Sector Time and Year

In [54]:
for year in range(2023, 2026):
    sector_times[year][f"TotalSectorTime_{year} (s)"]=(
        sector_times[year]["Sector1Time (s)"]+
        sector_times[year]["Sector2Time (s)"]+
        sector_times[year]["Sector3Time (s)"]
    )
    sector_times[year]["Year"] = year

In [55]:
sector_times[2024]

,Driver,Sector1Time (s),Sector2Time (s),Sector3Time (s),TotalSectorTime_2024 (s),Year
0,ALB,18.294725,32.249957,21.819913,72.364594,2024
1,ALO,18.713652,32.270536,21.922333,72.906522,2024
2,BOT,18.433464,32.215638,21.832855,72.481957,2024
3,GAS,18.131386,32.102671,21.843871,72.077929,2024
4,HAM,18.281700,31.798114,21.507329,71.587143,2024
5,HUL,18.218443,32.005843,21.753800,71.978086,2024
6,LEC,18.690686,31.859414,21.560886,72.110986,2024
7,MAG,18.123171,32.078443,21.860200,72.061814,2024
8,NOR,18.091159,31.711476,21.906619,71.709254,2024
9,OCO,18.251257,32.230729,21.709729,72.191714,2024


## Concating all the tables

In [56]:
combined_sector_time = pd.concat(sector_times.values(), ignore_index=True)

In [57]:
combined_sector_time

,Driver,Sector1Time (s),Sector2Time (s),Sector3Time (s),TotalSectorTime_2023 (s),Year,TotalSectorTime_2024 (s),TotalSectorTime_2025 (s)
0,ALB,18.579229,32.195086,22.230314,73.004629,2023,NaN,NaN
1,ALO,18.565886,31.831814,22.112986,72.510686,2023,NaN,NaN
2,BOT,18.798507,32.332826,22.410145,73.541478,2023,NaN,NaN
3,DEV,18.660580,32.326725,22.340609,73.327913,2023,NaN,NaN
4,GAS,18.757571,31.935757,22.148171,72.841500,2023,NaN,NaN
5,HAM,18.657243,31.985800,22.052229,72.695271,2023,NaN,NaN
6,HUL,21.874818,34.029273,25.448727,81.352818,2023,NaN,NaN
7,LEC,18.702700,31.715543,21.859271,72.277514,2023,NaN,NaN
8,MAG,18.811397,32.598985,22.355382,73.765765,2023,NaN,NaN
9,NOR,18.510629,31.966657,22.003100,72.480386,2023,NaN,NaN


## Clean Air Race Pace (Best Time From Practice Session Before Qualifying Round.)

In [58]:
clean_air_race_pace = {
    #McLaren
    "NOR":00,               #Lando Norris 🇬🇧
    "PIA":00,               #Oscar Piastri 🇦🇺
    #Red Bull Racing
    "VER":00,               #Max Verstappen 🇳🇱
    "HAD":00,               #Isack Hadjar 🇫🇷
    #Ferrari
    "LEC":00,               #Charles Leclerc 🇲🇨
    "HAM":00,               #Lewis Hamilton 🇬🇧
    #Mercedes
    "RUS":00,               #Geaorge Russell 🇬🇧
    "ANT":00,               #Andrea Kimi Antonelli 🇮🇹
    #Aston Martin Aramco
    "ALO":00,               #Fernando Alonso 🇪🇸
    "STR":00,               #Lance Stroll 🇨🇦
    #Williams
    "ALB":00,               #Alexander Albon 🇹🇭
    "SAI":00,               #Carlos Sainz Jr. 🇪🇸
    #Haas
    "OCO":00,               #Esteban Ocon 🇫🇷
    "BEA":00,               #Oliver Bearman 🇬🇧
    #Alpine
    "GAS":00,               #Piette Gasly 🇫🇷
    "COL":00,               #Franco Colapinto 🇦🇷
    #Audi
    "HUL":00,               #Nico Hulkenberg 🇩🇪
    "BOR":00,               #Gabriel Bortoleto 🇧🇷
    #Cadillac
    "PER":00,               #Sergio Perez 🇲🇽
    "BOT":00,               #Valtteri Bottas 🇫🇮
    #Racing Bulls
    "LAW":00,               #Liam Lawson 🇳🇿
    "LIN":00                #Arvid Lindblad 🇬🇧
}

## List of the Current Drivers

In [59]:
Driver = [
    #McLaren
    "NOR",               #Lando Norris 🇬🇧
    "PIA",               #Oscar Piastri 🇦🇺
    #Red Bull Racing
    "VER",               #Max Verstappen 🇳🇱
    "HAD",               #Isack Hadjar 🇫🇷
    #Ferrari
    "LEC",               #Charles Leclerc 🇲🇨
    "HAM",               #Lewis Hamilton 🇬🇧
    #Mercedes
    "RUS",               #Geaorge Russell 🇬🇧
    "ANT",               #Andrea Kimi Antonelli 🇮🇹
    #Aston Martin Aramco
    "ALO",               #Fernando Alonso 🇪🇸
    "STR",               #Lance Stroll 🇨🇦
    #Williams
    "ALB",               #Alexander Albon 🇹🇭
    "SAI",               #Carlos Sainz Jr. 🇪🇸
    #Haas
    "OCO",               #Esteban Ocon 🇫🇷
    "BEA",               #Oliver Bearman 🇬🇧
    #Alpine
    "GAS",               #Piette Gasly 🇫🇷
    "COL",               #Franco Colapinto 🇦🇷
    #Audi
    "HUL",               #Nico Hulkenberg 🇩🇪
    "BOR",               #Gabriel Bortoleto 🇧🇷
    #Cadillac
    "PER",               #Sergio Perez 🇲🇽
    "BOT",               #Valtteri Bottas 🇫🇮
    #Racing Bulls
    "LAW",               #Liam Lawson 🇳🇿
    "LIN"                #Arvid Lindblad 🇬🇧

]

## Qualifying Time 

In [60]:
qualifying_2026 = pd.DataFrame({
    "Driver":Driver,
    "QualifyingTime (s)":[
        #McLaren
        00,                 #NOR
        00,                 #PIA
        #Red Bull Racing
        00,                 #VER
        00,                 #HAD
        #Ferrari
        00,                 #LEC
        00,                 #HAM
        #Mercedes
        00,                 #RUS
        00,                 #ANT
        #Aston Martin Aramco
        00,                 #ALO
        00,                 #STR
        #Williams
        00,                 #ALB
        00,                 #SAI
        #Haas
        00,                 #OCO
        00,                 #BEA
        #Alpine
        00,                 #GAS
        00,                 #COL
        #Audi
        00,                 #HUL
        00,                 #BOR
        #Cadillac
        00,                 #PER
        00,                 #BOT
        #Racing Bulls
        00,                 #LAW
        00                  #LIN
    ]
})

## Combining Qualifying Time and Clean Air Race Pace 

In [61]:
qualifying_2026["CleanAirRacePace (s)"] = qualifying_2026["Driver"].map(clean_air_race_pace)

In [62]:
qualifying_2026

,Driver,QualifyingTime (s),CleanAirRacePace (s)
0,NOR,0,0
1,PIA,0,0
2,VER,0,0
3,HAD,0,0
4,LEC,0,0
5,HAM,0,0
6,RUS,0,0
7,ANT,0,0
8,ALO,0,0
9,STR,0,0


## Team Points(Currently Zero Due to Start of Season)

In [63]:
team_points = {
    "McLaren":00,
    "Mercedes":00,
    "Red Bull":00,
    "Williams":00,
    "Ferrari":00,
    "Haas":00,
    "Aston Martin":00,
    "Racing Bulls":00,
    "Alpine":00,
    "Audi":00,
    "Cardillac":00
}

## Scaling the Team Points (For Future Use)

In [64]:
from sklearn.preprocessing import MinMaxScaler

teams = list(team_points.keys())
points = np.array(list(team_points.values())).reshape(-1,1)

scaler = MinMaxScaler()
scaled_points = scaler.fit_transform(points)

team_performance_score = dict(zip(teams, scaled_points.flatten()))

## Mapping Drivers with their Respective Teams

In [65]:
driver_to_team = {
    "NOR":"McLaren",
    "PIA":"McLaren",

    "VER":"Red Bull",
    "HAD":"Red Bull",

    "LEC":"Ferrari",
    "HAM":"Ferrari",

    "RUS":"Mercedes",
    "ANT":"Mercedes",

    "ALO":"Aston Martin",
    "STR":"Aston Martin",

    "ALB":"Williams",
    "SAI":"Williams",

    "OCO":"Haas",
    "BEA":"Haas",

    "GAS":"Alpine",
    "COL":"Alpine",

    "HUL":"Audi",
    "BOR":"Audi",

    "PER":"Cardillac",
    "BOT":"Cardillac",
    
    "LAW":"Racing Bulls",
    "LIN":"Racing Bulls"  
}

## Combining Team and Team Performance data to Qualifying 2026 table

In [66]:
qualifying_2026["Team"] = qualifying_2026["Driver"].map(driver_to_team)
qualifying_2026["TeamPerformanceScore"] = qualifying_2026["Team"].map(team_performance_score)

In [67]:
qualifying_2026

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore
0,NOR,0,0,McLaren,0.0
1,PIA,0,0,McLaren,0.0
2,VER,0,0,Red Bull,0.0
3,HAD,0,0,Red Bull,0.0
4,LEC,0,0,Ferrari,0.0
5,HAM,0,0,Ferrari,0.0
6,RUS,0,0,Mercedes,0.0
7,ANT,0,0,Mercedes,0.0
8,ALO,0,0,Aston Martin,0.0
9,STR,0,0,Aston Martin,0.0


## Merging Data of qualifyingTime and Total Sector Time of 2023,2024,2025

In [72]:
merged_data = qualifying_2026.copy()

for year in range(2023, 2026):
    merged_data = merged_data.merge(
        sector_times[year][["Driver", f"TotalSectorTime_{year} (s)"]],
        on="Driver",
        how="left"
    )

In [73]:
merged_data

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s)
0,NOR,0,0,McLaren,0.0,72.480386,71.709254,71.577275
1,PIA,0,0,McLaren,0.0,73.628261,71.253629,71.589594
2,VER,0,0,Red Bull,0.0,72.228114,71.685729,NaN
3,HAD,0,0,Red Bull,0.0,NaN,NaN,72.625926
4,LEC,0,0,Ferrari,0.0,72.277514,72.110986,71.787116
5,HAM,0,0,Ferrari,0.0,72.695271,71.587143,71.883667
6,RUS,0,0,Mercedes,0.0,72.680000,71.262814,72.353652
7,ANT,0,0,Mercedes,0.0,NaN,NaN,NaN
8,ALO,0,0,Aston Martin,0.0,72.510686,72.906522,72.400044
9,STR,0,0,Aston Martin,0.0,72.875314,72.346377,72.814676


## Filling nan values of 2025 with the average values of 2023 and 2024

### This is done for the drivers whose data are present for 2023 and 2024 but where unable to perform in 2025 due to various reasons.

In [74]:
col_23 = "TotalSectorTime_2023 (s)"
col_24 = "TotalSectorTime_2024 (s)"
col_25 = "TotalSectorTime_2025 (s)"

mask = (
    merged_data[col_25].isna() &
    merged_data[col_23].notna() &
    merged_data[col_24].notna()
)

merged_data.loc[mask, col_25] = (
    merged_data.loc[mask, [col_23, col_24]].mean(axis=1)
)

In [75]:
merged_data

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s)
0,NOR,0,0,McLaren,0.0,72.480386,71.709254,71.577275
1,PIA,0,0,McLaren,0.0,73.628261,71.253629,71.589594
2,VER,0,0,Red Bull,0.0,72.228114,71.685729,71.956921
3,HAD,0,0,Red Bull,0.0,NaN,NaN,72.625926
4,LEC,0,0,Ferrari,0.0,72.277514,72.110986,71.787116
5,HAM,0,0,Ferrari,0.0,72.695271,71.587143,71.883667
6,RUS,0,0,Mercedes,0.0,72.680000,71.262814,72.353652
7,ANT,0,0,Mercedes,0.0,NaN,NaN,NaN
8,ALO,0,0,Aston Martin,0.0,72.510686,72.906522,72.400044
9,STR,0,0,Aston Martin,0.0,72.875314,72.346377,72.814676


## Adding a new column IsNewDriver to let the model know we have missing values and compute accordingly

In [76]:
merged_data["IsNewDriver"] = (
    merged_data["TotalSectorTime_2023 (s)"].isna() &
    merged_data["TotalSectorTime_2024 (s)"].isna()
).astype(int)

In [77]:
merged_data

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s),IsNewDriver
0,NOR,0,0,McLaren,0.0,72.480386,71.709254,71.577275,0
1,PIA,0,0,McLaren,0.0,73.628261,71.253629,71.589594,0
2,VER,0,0,Red Bull,0.0,72.228114,71.685729,71.956921,0
3,HAD,0,0,Red Bull,0.0,NaN,NaN,72.625926,1
4,LEC,0,0,Ferrari,0.0,72.277514,72.110986,71.787116,0
5,HAM,0,0,Ferrari,0.0,72.695271,71.587143,71.883667,0
6,RUS,0,0,Mercedes,0.0,72.680000,71.262814,72.353652,0
7,ANT,0,0,Mercedes,0.0,NaN,NaN,NaN,1
8,ALO,0,0,Aston Martin,0.0,72.510686,72.906522,72.400044,0
9,STR,0,0,Aston Martin,0.0,72.875314,72.346377,72.814676,0


## Imputing all the NAN values With Global Median

In [81]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

for year in range(2023, 2026):
    col = f"TotalSectorTime_{year} (s)"
    merged_data[[col]] = imputer.fit_transform(
        merged_data[[col]]
    )

In [82]:
merged_data

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s),IsNewDriver
0,NOR,0,0,McLaren,0.0,72.480386,71.709254,71.577275,0
1,PIA,0,0,McLaren,0.0,73.628261,71.253629,71.589594,0
2,VER,0,0,Red Bull,0.0,72.228114,71.685729,71.956921,0
3,HAD,0,0,Red Bull,0.0,72.695271,72.017786,72.625926,1
4,LEC,0,0,Ferrari,0.0,72.277514,72.110986,71.787116,0
5,HAM,0,0,Ferrari,0.0,72.695271,71.587143,71.883667,0
6,RUS,0,0,Mercedes,0.0,72.680000,71.262814,72.353652,0
7,ANT,0,0,Mercedes,0.0,72.695271,72.017786,72.376848,1
8,ALO,0,0,Aston Martin,0.0,72.510686,72.906522,72.400044,0
9,STR,0,0,Aston Martin,0.0,72.875314,72.346377,72.814676,0


In [83]:
merged_data.columns

Index(['Driver', 'QualifyingTime (s)', 'CleanAirRacePace (s)', 'Team',
       'TeamPerformanceScore', 'TotalSectorTime_2023 (s)',
       'TotalSectorTime_2024 (s)', 'TotalSectorTime_2025 (s)', 'IsNewDriver'],
      dtype='object')

# Lets try to predict 2025 using 2023 and 2024 data and make our model accurate

In [90]:
merged_data1 = merged_data.copy()

In [91]:
merged_data1

,Driver,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s),IsNewDriver
0,NOR,0,0,McLaren,0.0,72.480386,71.709254,71.577275,0
1,PIA,0,0,McLaren,0.0,73.628261,71.253629,71.589594,0
2,VER,0,0,Red Bull,0.0,72.228114,71.685729,71.956921,0
3,HAD,0,0,Red Bull,0.0,72.695271,72.017786,72.625926,1
4,LEC,0,0,Ferrari,0.0,72.277514,72.110986,71.787116,0
5,HAM,0,0,Ferrari,0.0,72.695271,71.587143,71.883667,0
6,RUS,0,0,Mercedes,0.0,72.680000,71.262814,72.353652,0
7,ANT,0,0,Mercedes,0.0,72.695271,72.017786,72.376848,1
8,ALO,0,0,Aston Martin,0.0,72.510686,72.906522,72.400044,0
9,STR,0,0,Aston Martin,0.0,72.875314,72.346377,72.814676,0


In [93]:
qualifying_2025 = pd.DataFrame({
    "Driver":Driver,
    "QualifyingTime (s)":[
        #McLaren
        63.971,                 #NOR 1:03.971
        64.492,                 #PIA 1:04.492
        #Red Bull Racing
        64.929,                 #VER 1:04.929
        65.226,                 #HAD 1:05.226
        #Ferrari
        64.492,                 #LEC 1:04.492
        64.582,                 #HAM 1:04.582
        #Mercedes
        64.763,                 #RUS 1:04.763
        65.276,                 #ANT 1:05.276
        #Aston Martin Aramco
        65.128,                 #ALO 1:05.128
        65.329,                 #STR 1:05.329
        #Williams
        65.205,                 #ALB 1:05.205
        65.582,                 #SAI 1:05.582
        #Haas
        65.364,                 #OCO 1:05.364
        65.312,                 #BEA 1:05.312
        #Alpine
        65.649,                 #GAS 1:05.649
        65.288,                 #COL 1:05.288
        #Audi
        65.606,                 #HUL 1:05.606
        65.132,                 #BOR 1:05.132
        #Cadillac
        00,                 #PER
        00,                 #BOT
        #Racing Bulls
        64.926,                 #LAW 1:04.926
        00                   #LIN
    ]
})


In [94]:
practice1 = fastf1.get_session(2025, 'Austria', 'FP1')
practice2 = fastf1.get_session(2025, 'Austria', 'FP2')
practice3 = fastf1.get_session(2025, 'Austria', 'FP3')

In [97]:
practice1.load()
practice2.load()
practice3.load()

core           INFO 	Loading data for Austrian Grand Prix - Practice 1 [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '5', '6', '10', '12', '14', '18', '22', '23', '27', '30', '31', '38', '43', '44', '55', '63', '81', '87', '89']
core           INFO 	Loading data for Austrian Grand Prix - Practice 2 [v3.8.1]
req            INFO 	No cac

In [100]:
practice1.laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 00:16:53.194000,VER,1,NaT,1.0,1.0,0 days 00:14:58.209000,NaT,NaT,0 days 00:00:49.665000,...,True,Red Bull Racing,0 days 00:14:58.209000,2025-06-27 11:31:04.784,1,NaN,False,,False,False
1,0 days 00:18:34.927000,VER,1,0 days 00:01:41.733000,2.0,1.0,NaT,NaT,0 days 00:00:16.970000,0 days 00:00:54.714000,...,True,Red Bull Racing,0 days 00:16:53.194000,2025-06-27 11:32:59.769,1,NaN,False,,False,True
2,0 days 00:19:42.594000,VER,1,0 days 00:01:07.667000,3.0,1.0,NaT,NaT,0 days 00:00:16.985000,0 days 00:00:30.216000,...,True,Red Bull Racing,0 days 00:18:34.927000,2025-06-27 11:34:41.502,1,NaN,False,,False,True
3,0 days 00:21:32.792000,VER,1,0 days 00:01:50.198000,4.0,1.0,NaT,0 days 00:21:29.462000,0 days 00:00:29.111000,0 days 00:00:49.464000,...,True,Red Bull Racing,0 days 00:19:42.594000,2025-06-27 11:35:49.169,1,NaN,False,,False,False
4,0 days 00:23:27.888000,VER,1,0 days 00:01:55.096000,5.0,2.0,0 days 00:21:54.601000,NaT,0 days 00:00:44.882000,0 days 00:00:46.724000,...,False,Red Bull Racing,0 days 00:21:32.792000,2025-06-27 11:37:39.367,1,NaN,False,,False,False


In [99]:
practice1.laps.columns

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')

In [107]:
plaps1 = practice1.laps.pick_quicklaps()

pbest_laps1 = plaps1.groupby("Driver")["LapTime"].min().dt.total_seconds()

print(pbest_laps1)

Driver
ALB    65.946
ALO    66.170
ANT    66.130
BEA    66.738
BEG    66.369
BOR    65.874
COL    66.246
DUN    65.766
GAS    65.780
HAD    66.110
HAM    66.099
HUL    66.140
LAW    66.189
OCO    66.510
PIA    65.697
RUS    65.542
SAI    66.017
STR    66.160
TSU    66.262
VER    65.607
Name: LapTime, dtype: float64


In [108]:
plaps2 = practice2.laps.pick_quicklaps()

pbest_laps2 = plaps2.groupby("Driver")["LapTime"].min().dt.total_seconds()

print(pbest_laps2)

Driver
ALB    65.765
ALO    65.457
ANT    65.537
BEA    65.835
BOR    65.411
COL    66.176
GAS    65.613
HAD    65.547
HAM    65.511
HUL    65.918
LAW    65.543
LEC    65.190
NOR    64.580
OCO    65.698
PIA    64.737
RUS    65.229
SAI    65.814
STR    65.022
TSU    65.292
VER    64.898
Name: LapTime, dtype: float64


In [109]:
plaps3 = practice3.laps.pick_quicklaps()

pbest_laps3 = plaps3.groupby("Driver")["LapTime"].min().dt.total_seconds()

print(pbest_laps3)

Driver
ALB    65.314
ALO    65.243
ANT    65.053
BEA    65.366
BOR    65.182
COL    65.546
GAS    65.366
HAD    66.023
HAM    64.790
HUL    65.283
LAW    65.182
LEC    64.574
NOR    64.324
OCO    65.519
PIA    64.442
RUS    65.018
SAI    65.326
STR    65.062
TSU    65.139
VER    64.534
Name: LapTime, dtype: float64


In [110]:
pbest_laps1 = pbest_laps1.rename("FP1")
pbest_laps2 = pbest_laps2.rename("FP2")
pbest_laps3 = pbest_laps3.rename("FP3")

combined = pd.concat(
    [pbest_laps1, pbest_laps2, pbest_laps3],
    axis=1
)


In [111]:
combined

,FP1,FP2,FP3
Driver,,,
ALB,65.946,65.765,65.314
ALO,66.170,65.457,65.243
ANT,66.130,65.537,65.053
BEA,66.738,65.835,65.366
BEG,66.369,NaN,NaN
BOR,65.874,65.411,65.182
COL,66.246,66.176,65.546
DUN,65.766,NaN,NaN
GAS,65.780,65.613,65.366


In [112]:
combined["Best_Practice_Time"] = combined.min(axis=1)

print(combined)

           FP1     FP2     FP3  Best_Practice_Time
Driver                                            
ALB     65.946  65.765  65.314              65.314
ALO     66.170  65.457  65.243              65.243
ANT     66.130  65.537  65.053              65.053
BEA     66.738  65.835  65.366              65.366
BEG     66.369     NaN     NaN              66.369
BOR     65.874  65.411  65.182              65.182
COL     66.246  66.176  65.546              65.546
DUN     65.766     NaN     NaN              65.766
GAS     65.780  65.613  65.366              65.366
HAD     66.110  65.547  66.023              65.547
HAM     66.099  65.511  64.790              64.790
HUL     66.140  65.918  65.283              65.283
LAW     66.189  65.543  65.182              65.182
OCO     66.510  65.698  65.519              65.519
PIA     65.697  64.737  64.442              64.442
RUS     65.542  65.229  65.018              65.018
SAI     66.017  65.814  65.326              65.326
STR     66.160  65.022  65.062 

In [114]:
clean_air_race_pace_2025 = {
    #McLaren
    "NOR":64.324,               #Lando Norris 🇬🇧
    "PIA":64.442,               #Oscar Piastri 🇦🇺
    #Red Bull Racing
    "VER":64.534,               #Max Verstappen 🇳🇱
    "HAD":65.547,               #Isack Hadjar 🇫🇷
    #Ferrari
    "LEC":00,               #Charles Leclerc 🇲🇨
    "HAM":64.790,               #Lewis Hamilton 🇬🇧
    #Mercedes
    "RUS":65.018,               #Geaorge Russell 🇬🇧
    "ANT":65.053,               #Andrea Kimi Antonelli 🇮🇹
    #Aston Martin Aramco
    "ALO":65.243,               #Fernando Alonso 🇪🇸
    "STR":65.022,               #Lance Stroll 🇨🇦
    #Williams
    "ALB":65.243,               #Alexander Albon 🇹🇭
    "SAI":65.326,               #Carlos Sainz Jr. 🇪🇸
    #Haas
    "OCO":65.519,               #Esteban Ocon 🇫🇷
    "BEA":65.366,               #Oliver Bearman 🇬🇧
    #Alpine
    "GAS":65.366,               #Piette Gasly 🇫🇷
    "COL":65.546,               #Franco Colapinto 🇦🇷
    #Audi
    "HUL":65.283,               #Nico Hulkenberg 🇩🇪
    "BOR":65.182,               #Gabriel Bortoleto 🇧🇷
    #Cadillac
    "PER":00,               #Sergio Perez 🇲🇽
    "BOT":00,               #Valtteri Bottas 🇫🇮
    #Racing Bulls
    "LAW":65.182,               #Liam Lawson 🇳🇿
    "LIN":00                #Arvid Lindblad 🇬🇧
}

In [116]:
team_points_2025 = {
    "McLaren":374,
    "Mercedes":199,
    "Red Bull":162,
    "Williams":55,
    "Ferrari":183,
    "Haas":28,
    "Aston Martin":22,
    "Racing Bulls":28,
    "Alpine":11,
    "Audi":00,
    "Cardillac":00
}

In [118]:
from sklearn.preprocessing import MinMaxScaler

teams_2025 = list(team_points_2025.keys())
points_2025 = np.array(list(team_points_2025.values())).reshape(-1,1)

scaler = MinMaxScaler()
scaled_points_2025 = scaler.fit_transform(points_2025)

team_performance_score_2025 = dict(zip(teams_2025, scaled_points_2025.flatten()))

In [119]:
qualifying_2025["Team"] = qualifying_2025["Driver"].map(driver_to_team)
qualifying_2025["TeamPerformanceScore"] = qualifying_2025["Team"].map(team_performance_score_2025)

In [128]:
qualifying_2025

,QualifyingTime (s),Team,TeamPerformanceScore
Driver,,,
NOR,63.971,McLaren,1.000000
PIA,64.492,McLaren,1.000000
VER,64.929,Red Bull,0.433155
HAD,65.226,Red Bull,0.433155
LEC,64.492,Ferrari,0.489305
HAM,64.582,Ferrari,0.489305
RUS,64.763,Mercedes,0.532086
ANT,65.276,Mercedes,0.532086
ALO,65.128,Aston Martin,0.058824


In [129]:
merged_data1

,QualifyingTime (s),CleanAirRacePace (s),Team,TeamPerformanceScore,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s),IsNewDriver
Driver,,,,,,,,
NOR,0,0,McLaren,0.0,72.480386,71.709254,71.577275,0
PIA,0,0,McLaren,0.0,73.628261,71.253629,71.589594,0
VER,0,0,Red Bull,0.0,72.228114,71.685729,71.956921,0
HAD,0,0,Red Bull,0.0,72.695271,72.017786,72.625926,1
LEC,0,0,Ferrari,0.0,72.277514,72.110986,71.787116,0
HAM,0,0,Ferrari,0.0,72.695271,71.587143,71.883667,0
RUS,0,0,Mercedes,0.0,72.680000,71.262814,72.353652,0
ANT,0,0,Mercedes,0.0,72.695271,72.017786,72.376848,1
ALO,0,0,Aston Martin,0.0,72.510686,72.906522,72.400044,0


In [ ]:
merged_data1 = merged_data1.set_index("Driver")
qualifying_2025 = qualifying_2025.set_index("Driver")

merged_data1[["QualifyingTime (s)", "Team", "TeamPerformanceScore"]] = qualifying_2025[["QualifyingTime (s)", "Team", "TeamPerformanceScore"]]

KeyError: "None of ['Driver'] are in the columns"

In [124]:
final_df

,Driver,QualifyingTime (s)_x,Team_x,TeamPerformanceScore_x,QualifyingTime (s)_y,CleanAirRacePace (s),Team_y,TeamPerformanceScore_y,TotalSectorTime_2023 (s),TotalSectorTime_2024 (s),TotalSectorTime_2025 (s),IsNewDriver
0,NOR,63.971,McLaren,1.000000,0,0,McLaren,0.0,72.480386,71.709254,71.577275,0
1,PIA,64.492,McLaren,1.000000,0,0,McLaren,0.0,73.628261,71.253629,71.589594,0
2,VER,64.929,Red Bull,0.433155,0,0,Red Bull,0.0,72.228114,71.685729,71.956921,0
3,HAD,65.226,Red Bull,0.433155,0,0,Red Bull,0.0,72.695271,72.017786,72.625926,1
4,LEC,64.492,Ferrari,0.489305,0,0,Ferrari,0.0,72.277514,72.110986,71.787116,0
5,HAM,64.582,Ferrari,0.489305,0,0,Ferrari,0.0,72.695271,71.587143,71.883667,0
6,RUS,64.763,Mercedes,0.532086,0,0,Mercedes,0.0,72.680000,71.262814,72.353652,0
7,ANT,65.276,Mercedes,0.532086,0,0,Mercedes,0.0,72.695271,72.017786,72.376848,1
8,ALO,65.128,Aston Martin,0.058824,0,0,Aston Martin,0.0,72.510686,72.906522,72.400044,0
9,STR,65.329,Aston Martin,0.058824,0,0,Aston Martin,0.0,72.875314,72.346377,72.814676,0


## Defining Features (X) and Target (y)

In [ ]:
X = merged_data1[[
    'QualifyingTime (s)', 'CleanAirRacePace (s)', 
       'TeamPerformanceScore', 'IsNewDriver'
]]

In [89]:
X

,QualifyingTime (s),CleanAirRacePace (s),TeamPerformanceScore,IsNewDriver
0,0,0,0.0,0
1,0,0,0.0,0
2,0,0,0.0,0
3,0,0,0.0,1
4,0,0,0.0,0
5,0,0,0.0,0
6,0,0,0.0,0
7,0,0,0.0,1
8,0,0,0.0,0
9,0,0,0.0,0


In [ ]:
y = 

In [87]:
y

Driver
NOR    71.709254
PIA    71.253629
VER    71.685729
HAD          NaN
LEC    72.110986
HAM    71.587143
RUS    71.262814
ANT          NaN
ALO    72.906522
STR    72.346377
ALB    72.364594
SAI    71.312500
OCO    72.191714
BEA          NaN
GAS    72.077929
COL          NaN
HUL    71.978086
BOR          NaN
PER    72.017786
BOT    72.481957
LAW          NaN
LIN          NaN
Name: LapTime (s), dtype: float64